# NeuroScan Agent: end-to-end training on Colab
Runtime → Change runtime type → **GPU**. Everything is saved to Google Drive so it survives disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/neuroscan'
!mkdir -p $DRIVE/data/raw/kaggle $DRIVE/data/raw/figshare $DRIVE/checkpoints $DRIVE/results

In [ ]:
# Replace with your repo URL
!git clone https://github.com/hannay-sen/NEUROSCAN.git /content/neuroscan
%cd /content/neuroscan
!pip install -q -r requirements.txt && pip install -q -e .

## Link data, checkpoints and results to Drive
Put the Kaggle *Brain Tumor MRI Dataset* folders (`Training/`, `Testing/`) in `MyDrive/neuroscan/data/raw/kaggle/` and the Figshare `.mat` files in `MyDrive/neuroscan/data/raw/figshare/`.

In [ ]:
!rm -rf data checkpoints results
!ln -s $DRIVE/data data
!ln -s $DRIVE/checkpoints checkpoints
!ln -s $DRIVE/results results
!ls data/raw/kaggle data/raw/figshare | head

## (Optional) download Kaggle data with the Kaggle API
Upload `kaggle.json` from your Kaggle account settings first.

In [ ]:
# from google.colab import files; files.upload()   # kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p data/raw/kaggle --unzip

## 1. Data

In [ ]:
!python -m neuroscan.data.prepare_figshare
!python -m neuroscan.data.dedup
!python -m neuroscan.data.make_splits

In [ ]:
import pandas as pd
pd.read_csv('data/splits/dedup_pairs.csv').sample(10, random_state=0)   # spot-check matches

## 2. Train

In [ ]:
!python -m neuroscan.train_classifier
!python -m neuroscan.train_classifier --balance none --tag nobal

In [ ]:
!python -m neuroscan.train_segmentation

## 3. Evaluate

In [ ]:
!python -m neuroscan.eval.evaluate_classifier --split kaggle_test
!python -m neuroscan.eval.evaluate_classifier --split figshare_all
!python -m neuroscan.eval.evaluate_classifier --split kaggle_test --tag nobal
!python -m neuroscan.eval.evaluate_segmentation --split figshare_test
!python -m neuroscan.eval.evaluate_safety --split kaggle_val --sweep

Copy the chosen `min_confidence` / `max_entropy` into `configs/config.yaml`, then:

In [ ]:
!python -m neuroscan.eval.evaluate_safety --split kaggle_test
!ls results

## 4. Try one scan

In [ ]:
import glob
!python -m neuroscan.graph {glob.glob('data/raw/kaggle/Testing/glioma/*')[0]}